# Tutorial: Multi-area Laminar Models (JAX/Jaxfne)

Heavy computation lives in `jaxfne`; this notebook exposes the controls. Everything runs through the real package engine (`import jaxfne as jtfne`) — no local simulator

**Learning objectives**

1. Explore **E / PV / SST / VIP** Izhikevich waveforms and understand how `a, b, c, d, drive` shape action-potential width, frequency content, and excitatory vs inhibitory identity.
2. Define a **two-area, 6-layer, 4-cell-type** laminar network with editable within-area and between-area (feedforward / feedback) connectivity.
3. Apply **AGSDR fine-tuning** to drive a target firing rate while minimising spike synchrony (kappa).
4. Inject **custom stimuli** into any cell population and apply **lesions** to any layer, cell type, or area.
5. Manipulate **spectrolaminar parameters** — cell-type ratios, densities, drives, and waveforms — and observe how alpha-beta (deep) vs gamma (superficial) relative-power profiles respond.
6. Run **knock-out experiments** — silence a layer or cell type and measure the downstream effect on per-area firing rates.
7. Inspect and export **numeric spectrolaminar motif metrics** — band-peak depths, crossing detection, finite-array checks, and a JSON-safe manifest and validation report.

**Scope / status:** simulated **proxy** readouts in a computational scaffold (`laminar_proxy_no_pde`). The spectrolaminar motif is **emergent** from the dynamics + a depth-dependent leadfield — it is *not* imposed.

**NOTE: **

## 1. Install and import

In [ ]:
#@title Install jaxfne
import os

INSTALL_MODE = "github-main"  #@param ["pypi", "github-main", "local-editable", "skip"]

# Local validation (PYTHONPATH=. with TFNE_SMOKE=1) must use the checked-out code.
if os.environ.get("TFNE_SMOKE") == "1":
    INSTALL_MODE = "skip"

if INSTALL_MODE == "pypi":
    %pip install -q "jaxfne[viz]"
elif INSTALL_MODE == "github-main":
    %pip install -q "jaxfne[viz] @ git+https://github.com/HNXJ/jaxfne.git@main"
elif INSTALL_MODE == "local-editable":
    %pip install -q -e .
elif INSTALL_MODE == "skip":
    pass

# On Colab GPU, make sure JAX has CUDA wheels.
# JAX docs recommend the pip CUDA/cUDNN wheels for NVIDIA GPU installs.
# We do this AFTER installing jaxfne to fix any jax/jaxlib version mismatches.
JAX_BACKEND = "gpu"

if JAX_BACKEND == "gpu" and "COLAB_GPU" in os.environ:
    %pip install -q -U "jax[cuda12]"

import jax
import numpy as np
import jaxfne as jtfne


print("jaxfne:", getattr(jtfne, "__version__", "unknown"))
print("jax:", jax.__version__)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())

## 2. Editable run size, duration, and sampling

Choose `RUN_MODE`. Smoke mode is forced by `TFNE_SMOKE=1` for CI/local validation. Use `release` for the strict 1000 ms / 0.1 ms run; use `interactive` while editing ratios and geometry.

In [ ]:
#@title Runtime / scale controls
SMOKE = os.environ.get("TFNE_SMOKE") == "1"
RUN_MODE = "interactive"  #@param ["interactive", "release"]
if SMOKE:
    RUN_MODE = "smoke"

# A clean spectrolaminar motif needs >= 10 trials and >= 32 equally spaced
# contacts; interactive/release use those, smoke stays tiny for fast CI.
SCALE_BY_MODE = {
    "smoke":       dict(n_neuron_per_column=12,  duration_ms=40.0,   dt_ms=0.05,
                        n_trials=2,  n_contacts=8,  freq_count=32),
    "interactive": dict(n_neuron_per_column=500, duration_ms=1000.0,  dt_ms=0.05,
                        n_trials=5, n_contacts=128, freq_count=32),
    "release":     dict(n_neuron_per_column=500, duration_ms=1000.0, dt_ms=0.05,
                        n_trials=5, n_contacts=128, freq_count=32),
}

sizing = SCALE_BY_MODE[RUN_MODE]
print("RUN_MODE:", RUN_MODE)
print(sizing)


## 3. Editable column dimensions

Edit these geometry anchors directly. Units are meters. `COLUMN_Z_M` sets cortical depth; `COLUMN_X_M` and `COLUMN_Y_M` set the area spacing scale used by the scaffold visualization.

In [ ]:
#@title Column geometry controls
COLUMN_X_M = 1.0e-3  # x spacing anchor
COLUMN_Y_M = 1.0e-3  # y spacing anchor
COLUMN_Z_M = 2.0e-3  # cortical depth anchor
COLUMN_RADIUS_REL = 0.10  # radius as fraction of min(x, y)
L4_REF_REL = 0.50  # L4 reference as fraction of depth

GEOMETRY_KWARGS = dict(
    cx_m=COLUMN_X_M,
    cy_m=COLUMN_Y_M,
    cz_m=COLUMN_Z_M,
    radius_rel=COLUMN_RADIUS_REL,
    l4_ref_rel=L4_REF_REL,
)
GEOMETRY_KWARGS


## 4. Editable layers and layer sizes

Layer boundaries are fractions of `COLUMN_Z_M`. `LAYER_COUNT_FRAC` controls how many neurons are allocated to each layer. Values are normalized by the config helper.

In [ ]:
LAYERS = ("L1", "L2", "L3", "L4", "L5", "L6")
LAYER_FRACTIONS = (
    ("L1", 0.00, 0.08),
    ("L2", 0.08, 0.20),
    ("L3", 0.20, 0.35),
    ("L4", 0.35, 0.45),
    ("L5", 0.45, 0.75),
    ("L6", 0.75, 1.00),
)
# Cortically realistic neuron density: L5 is the thickest layer and most
# E-dominant, producing the bulk of the slow (alpha-beta) signal. L1 is thin
# and sparsely populated. L4 is the thalamorecipient input layer.
LAYER_COUNT_FRAC = {
    "L1": 0.08,
    "L2": 0.12,
    "L3": 0.15,
    "L4": 0.10,
    "L5": 0.30,
    "L6": 0.25,
}


## 5. Editable cell-type ratios per layer

This is the main cell-distribution matrix. Rows are layers, columns are cell types. Edit the numbers; rows do **not** need to sum to 1 because `make_cell_dist` normalizes each row.

For example, if `CELL_TYPES=("E", "PV")` and six layers are listed, `cfg.cell_dist.shape == (6, 2)`.

In [ ]:
CELL_TYPES = ("E", "PV", "SST", "VIP")
CELL_COLORS = {"E": "#e69500", "PV": "#0072ce", "SST": "#ffbf00", "VIP": "#7b3294"}
CELL_SIGNS = {"E": 1.0, "PV": -1.0, "SST": -1.0, "VIP": -1.0}

# ── Internal DRIVE per cell type (d1, d2, d3, d4) — the main excitability knobs ─
# Tuned so the network stays STABLE: average ~5-10 Hz/neuron, peak < ~40 Hz.
# Raise a drive to make that population more active (knock-in / rescue / probe);
# lower toward 0 to quieten it. These are native/uncalibrated proxy units.
E_DRIVE, PV_DRIVE, SST_DRIVE, VIP_DRIVE = 2.0, 2.0, 1.0, 2.0

# ── Internal NOISE per cell type — stochastic-current std (stability knob) ─────
# 0.0 = deterministic; higher = more irregular firing. Independent per type.
E_NOISE, PV_NOISE, SST_NOISE, VIP_NOISE = 2.0, 2.0, 2.0, 2.0

# Full per-cell-type Izhikevich parameters. Waveform shape = (a, b, c, d);
# excitability/irregularity = (drive, noise) from the knobs above. Wider/slower E
# (low a, high c/d) -> alpha-beta; fast-spiking PV -> gamma. VIP uses b=+0.20 so
# it fires at modest drive (the b=-0.10 IS profile has rheobase ~23).
CELL_TYPE_IZH = {
    "E":   {"a": 0.015, "b": 0.20, "c": -60.0, "d": 10.0, "drive": E_DRIVE,   "noise": E_NOISE},
    "PV":  {"a": 0.10,  "b": 0.20, "c": -65.0, "d": 2.0,  "drive": PV_DRIVE,  "noise": PV_NOISE},
    "SST": {"a": 0.02,  "b": 0.25, "c": -65.0, "d": 2.0,  "drive": SST_DRIVE, "noise": SST_NOISE},
    "VIP": {"a": 0.02,  "b": 0.20, "c": -55.0, "d": 6.0,  "drive": VIP_DRIVE, "noise": VIP_NOISE},
}

# Layer composition — E maximal DEEP (L5/L6), inhibitory enriched SUPERFICIAL
# (L1-L3), L4 thalamorecipient (PV-gated). Edit freely.
LAYER_CELL_TYPE_FRAC = {
    "L1": {"E": 0.20, "PV": 0.10, "SST": 0.10, "VIP": 0.60},
    "L2": {"E": 0.40, "PV": 0.15, "SST": 0.30, "VIP": 0.15},
    "L3": {"E": 0.40, "PV": 0.35, "SST": 0.20, "VIP": 0.05},
    "L4": {"E": 0.50, "PV": 0.35, "SST": 0.10, "VIP": 0.05},
    "L5": {"E": 0.90, "PV": 0.05, "SST": 0.04, "VIP": 0.01},
    "L6": {"E": 0.90, "PV": 0.05, "SST": 0.04, "VIP": 0.01},
}

cell_dist_preview = jtfne.tutorial_utils.make_cell_dist(
    LAYERS, CELL_TYPES, LAYER_CELL_TYPE_FRAC
)
try:
    import pandas as pd
    display(pd.DataFrame(cell_dist_preview, index=LAYERS, columns=CELL_TYPES).style.format("{:.2f}"))
except ImportError:
    print(cell_dist_preview)


## 5b. Editable inter-area connectivity and lesions

`CONNECTIVITY_SPEC` lists **between-area** projection rules added on top of within-area recurrence. Each rule matches presynaptic source neurons (`src_area` / `src_layers` / `src_cell_types`) to postsynaptic destinations, with a `weight` (proxy units) and an optional `control_key` whose value in `base_control` scales it. Defaults are the canonical hierarchy: **feedforward** V1 L2/3 (E) → V4 L4 and **feedback** V4 L2/3 (E) → V1 L5/6. (Thalamic input → L4 of V1 is the *stimulus* target, set later.)

`LESION_SPEC` silences matching neurons for knock-out experiments (leave empty for an intact network).


In [ ]:
# Inter-area projections (edit / extend freely). weight is in proxy units.
CONNECTIVITY_SPEC = (
    {"name": "feedforward", "src_area": "V1", "src_layers": ("L2", "L3"), "src_cell_types": ("E",),
     "dst_area": "V4", "dst_layers": ("L4",), "dst_cell_types": None,
     "weight": 5.0, "control_key": "feedforward_gain"},
    {"name": "feedback", "src_area": "V4", "src_layers": ("L2", "L3"), "src_cell_types": ("E",),
     "dst_area": "V1", "dst_layers": ("L5", "L6"), "dst_cell_types": None,
     "weight": 5.0, "control_key": "feedback_gain"},
    # Local pathways with increased gains
    {"name": "L4_to_L23_V1", "src_area": "V1", "src_layers": ("L4",), "src_cell_types": ("E",),
     "dst_area": "V1", "dst_layers": ("L2", "L3"), "dst_cell_types": None,
     "weight": 12.0, "control_key": "feedforward_gain"},
    {"name": "L4_to_L23_V4", "src_area": "V4", "src_layers": ("L4",), "src_cell_types": ("E",),
     "dst_area": "V4", "dst_layers": ("L2", "L3"), "dst_cell_types": None,
     "weight": 12.0, "control_key": "feedforward_gain"},
    {"name": "deep_to_L23_V1", "src_area": "V1", "src_layers": ("L5", "L6"), "src_cell_types": ("E",),
     "dst_area": "V1", "dst_layers": ("L2", "L3"), "dst_cell_types": None,
     "weight": 12.0, "control_key": "feedback_gain"},
    {"name": "deep_to_L23_V4", "src_area": "V4", "src_layers": ("L5", "L6"), "src_cell_types": ("E",),
     "dst_area": "V4", "dst_layers": ("L2", "L3"), "dst_cell_types": None,
     "weight": 12.0, "control_key": "feedback_gain"},
)


## 6. Build the jaxfne configuration

All editable controls above are passed into `jtfne.tutorial_utils.make_laminar_column_config`. The gates below keep the matrix shape, normalization, and proxy-scope status explicit.

In [ ]:
cfg = jtfne.tutorial_utils.make_laminar_column_config(
    areas=("V1", "V4"),          # two-area hierarchy (extend, e.g. ("V1","V4","PFC"))
    area_x_rel=(-1.0, 1.0),
    layers=LAYERS,
    cell_types=CELL_TYPES,
    layer_fractions=LAYER_FRACTIONS,
    layer_count_frac=LAYER_COUNT_FRAC,
    layer_cell_type_frac=LAYER_CELL_TYPE_FRAC,
    cell_colors=CELL_COLORS,
    cell_signs=CELL_SIGNS,
    cell_type_izh_params=CELL_TYPE_IZH,
    connectivity_spec=CONNECTIVITY_SPEC,
    lesion_spec=LESION_SPEC,
    seed=20260512,
    output_dir="outputs/jaxfne_etude_no_1",
    **GEOMETRY_KWARGS,
    **sizing,
)

assert cfg.cell_dist.shape == (len(cfg.layers), len(cfg.cell_types))
assert np.allclose(cfg.cell_dist.sum(axis=1), 1.0)
assert cfg.truth_gates["truth_mode"] == "truth_safe_unverified"
assert cfg.truth_gates["field_solver_status"] == "laminar_proxy_no_pde"
assert cfg.cell_type_izh_params["E"]["a"] < 0.02  # wider/slower E action potential

display(cfg.cell_dist_frame)
display(jtfne.tutorial_utils.config_summary_frame(cfg))


## 7. Izhikevich control panel

In [ ]:
panel = jtfne.tutorial_utils.make_izhikevich_control_panel(cfg, preset="PV")
control = jtfne.tutorial_utils.collect_izhikevich_control(panel)
if panel is not None:
    from IPython.display import display
    display(panel)
else:
    print("ipywidgets unavailable; continuing with default control values.")


### 7b. Explore E / PV / SST / VIP waveforms

Simulate each cell type as a single isolated Izhikevich emitter to see its action-potential shape. These are the exact per-type dynamics that go into the network, so editing `CELL_TYPE_IZH` above changes both these waveforms and the spectrolaminar result. A **wider / slower** E AP (lower `a`, higher `c`/`d`) carries more low-frequency (alpha-beta) power; fast-spiking PV pushes gamma.

In [ ]:
WAVEFORM_DRIVE = 10.0  # constant injected current for the isolated-cell APs

waveforms = jtfne.tutorial_utils.single_cell_waveforms(
    cfg, cell_types=cfg.cell_types, duration_ms=200.0, dt_ms=0.25,
    drive=WAVEFORM_DRIVE, cell_type_izh_params=CELL_TYPE_IZH,
)

try:
    import matplotlib.pyplot as plt
    fig_wf, axes_wf = plt.subplots(1, len(waveforms), figsize=(3.2 * len(waveforms), 3.0), sharey=True)
    axes_wf = np.atleast_1d(axes_wf)
    for ax, (ct, w) in zip(axes_wf, waveforms.items()):
        ax.plot(w["time_ms"], w["voltage_mV"], color=CELL_COLORS.get(ct, "#444"), lw=1.2)
        sign = "exc" if CELL_SIGNS.get(ct, 1.0) > 0 else "inh"
        ax.set_title(f"{ct} ({sign})\n{int(w['spikes'].sum())} spikes", fontsize=9)
        ax.set_xlabel("Time (ms)")
    axes_wf[0].set_ylabel("V (proxy mV)")
    fig_wf.suptitle(f"Single-cell Izhikevich waveforms (drive={WAVEFORM_DRIVE})", fontsize=11)
    fig_wf.tight_layout()
    plt.show()
except ImportError:
    for ct, w in waveforms.items():
        print(ct, "spikes:", int(w["spikes"].sum()), "params:", w["params"])


## 8. Build model and 3D cortical scaffold

In [ ]:
# Build the scaffold if this cell is run after a runtime restart.
if "model" not in globals():
    model = jtfne.tutorial_utils.build_laminar_column(cfg)
print("neurons:", len(model["neurons"]), "| areas:", cfg.areas)

fig_net = jtfne.vis.visualize_network_3d(
    model["neurons"],
    title="V1-V4-PFC TFNE-Izhikevich cortical scaffold",
    coordinate_unit="m",
    display_unit="um",
    show_column_shells=True,
    column_shape="cylinder",
    cell_type_colors={
        "E": "#e6a000",
        "PV": "#17a2c4",
        "SST": "#e000c8",
        "VIP": "#d0d0d0",
    },
    output_html=f"{cfg.output_dir}/scaffold_3d.html",
)

# Dark theme + fixed 3-D view controls
fig_net.update_layout(
    template="plotly_dark",
    paper_bgcolor="black",
    plot_bgcolor="black",
    font=dict(color="white"),
    title=dict(
        text="V1-V4-PFC TFNE-Izhikevich cortical scaffold",
        x=0.5,
        font=dict(color="white", size=18),
    ),
    scene=dict(
        bgcolor="black",

        # Axis ranges: adjust these to zoom/crop in displayed units.
        # Because display_unit="um", these are usually micrometers.
        xaxis=dict(
            title="x (um)",
            backgroundcolor="black",
            gridcolor="#333333",
            zerolinecolor="#666666",
            color="white",
            # range=[-400, 400],
        ),
        yaxis=dict(
            title="y (um)",
            backgroundcolor="black",
            gridcolor="#333333",
            zerolinecolor="#666666",
            color="white",
            # range=[-400, 400],
        ),
        zaxis=dict(
            title="depth (um)",
            backgroundcolor="black",
            gridcolor="#333333",
            zerolinecolor="#666666",
            color="white",
            # range=[200, -1200],
        ),

        # Axis aspect scaling.
        # x=1,y=1 keeps circular columns circular.
        # Increase z to visually stretch cortical depth.
        aspectmode="manual",
        aspectratio=dict(x=1.0, y=1.0, z=3.2),

        # Initial camera rotation / zoom.
        camera=dict(
            eye=dict(x=1.7, y=-1.8, z=-1.2),
            center=dict(x=0.0, y=0.0, z=0.0),
            up=dict(x=0.0, y=0.0, z=1.0),
        ),
    ),
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=50),
)

# Important: visualize_network_3d already wrote HTML before the styling update.
# Write it again so the saved HTML is dark too.
fig_net.write_html(f"{cfg.output_dir}/scaffold_3d.html")

fig_net

## 9. Editable stimulus, target cells, and simulation

Change `STIMULUS_KIND`, frequency, amplitude, target area/layers/types, and target fraction. This is the simplest place to test a new perturbation.

In [ ]:
STIMULUS_KIND = "sine"  # constant, sine, step, pulses, noise
STIMULUS_AMPLITUDE = 2.5
STIMULUS_FREQUENCY_HZ = 1.0
TARGET_AREA = "V1"
TARGET_LAYERS = ("L4",)
TARGET_CELL_TYPES = ("E",)
TARGET_FRACTION = 1.0

stimulus = jtfne.tutorial_utils.make_stimulus(
    kind=STIMULUS_KIND,
    duration_ms=cfg.duration_ms,
    dt_ms=cfg.dt_ms,
    amplitude=STIMULUS_AMPLITUDE,
    frequency_hz=STIMULUS_FREQUENCY_HZ,
    seed=cfg.seed,
)

target_cells = jtfne.tutorial_utils.select_cells(
    model, area=TARGET_AREA, layers=TARGET_LAYERS,
    cell_types=TARGET_CELL_TYPES, fraction=TARGET_FRACTION, seed=cfg.seed
)

trials = jtfne.tutorial_utils.simulate_laminar_trials(
    model, cfg, cfg.base_control, stimulus, target_cells, n_trials=cfg.n_trials
)

for k in ("spikes", "voltage_mV", "source_native"):
    assert trials[k].ndim == 3
for k in ("lfp_contacts", "csd_contacts"):
    assert trials[k].ndim == 4 and trials[k].shape[1] == len(cfg.areas)
    assert np.isfinite(trials[k]).all()
print({k: tuple(np.asarray(trials[k]).shape) for k in
       ("spikes", "voltage_mV", "source_native", "lfp_contacts", "csd_contacts")})


### 9b. Linear-distance LFP proxy with extended cortical coverage

This cell reprojects the package source traces onto laminar contacts using a **linear inverse-distance depth metric** instead of the default Gaussian leadfield. Contacts span from **20% above L1** to **20% below L6**, so the spectrolaminar readout covers the cortex plus a margin on both sides. This remains a simulated proxy readout under `laminar_proxy_no_pde`; it is not a solved volume-conductor field.


In [ ]:
#@title Linear-distance LFP proxy for spectrolaminar motif
# This replaces trials["lfp_contacts"], trials["csd_contacts"], and
# trials["contact_depths_m"] using the real source_native traces from the
# Izhikevich simulation. It changes only the laminar readout/probe projection;
# spikes, voltage, and native sources stay unchanged.

LFP_DEPTH_EXTENSION_FRAC = 0.20  # contacts cover [-20%, 120%] of cortical depth
LFP_DISTANCE_FLOOR_FRAC = 0.025  # avoids singular weights at exactly matched depth


def reproject_trials_lfp_linear_distance(
    trials,
    model,
    cfg,
    *,
    extension_frac=0.20,
    distance_floor_frac=0.025,
):
    """Recompute LFP/CSD proxy contacts with an extended linear-distance kernel.

    Source tensor:
        trials["source_native"] has shape (trial, time, neuron).

    Contact axis:
        contacts_rel = linspace(-extension_frac, 1 + extension_frac, n_contacts)
        so extension_frac=0.20 covers 20% above L1 and 20% below L6.

    Linear inverse-distance kernel:
        distance[c,n] = abs(contact_rel[c] - neuron_depth_rel[n])
        kernel[c,n]   = 1 / (distance[c,n] + distance_floor_frac)
        kernel[c,:]   = kernel[c,:] / sum_n(kernel[c,n])

    Output tensor contract:
        lfp_contacts, csd_contacts: (trial, area, time, contact)
        contact_depths_m:           (contact,)

    Scope/status:
        This is still a row-normalized laminar proxy readout. It is not a
        Poisson/Maxwell/volume-conductor solve and carries no calibrated
        physical amplitude claim.
    """
    source = np.asarray(trials["source_native"], dtype=np.float32)
    if source.ndim != 3:
        raise ValueError(f"source_native must be (trial,time,neuron), got {source.shape}")
    if not np.all(np.isfinite(source)):
        raise ValueError("source_native contains non-finite values")

    neurons = model["neurons"]
    areas = tuple(trials.get("area_names", cfg.areas))
    n_trials, n_steps, n_neurons = source.shape
    n_contacts = int(cfg.n_contacts)
    cz_m = float(cfg.cz_m)
    if not np.isfinite(cz_m) or cz_m <= 0:
        raise ValueError(f"invalid cfg.cz_m: {cz_m}")

    contacts_rel = np.linspace(
        -float(extension_frac),
        1.0 + float(extension_frac),
        n_contacts,
        dtype=np.float64,
    )
    contact_depths_m = (contacts_rel * cz_m).astype(np.float32)
    dz_rel = float(contacts_rel[1] - contacts_rel[0]) if n_contacts > 1 else 1.0

    lfp_contacts = np.zeros((n_trials, len(areas), n_steps, n_contacts), dtype=np.float32)
    csd_contacts = np.zeros_like(lfp_contacts)
    row_sum_errors = []
    kernel_stats = {}

    area_values = neurons["area"].to_numpy()
    z_m_all = neurons["z_m"].to_numpy(dtype=np.float64)
    if z_m_all.shape[0] != n_neurons:
        raise ValueError("model neuron count does not match source_native neuron axis")

    for ai, area in enumerate(areas):
        idx = np.flatnonzero(area_values == area)
        if idx.size == 0:
            continue

        z_rel = np.clip(z_m_all[idx] / cz_m, 0.0, 1.0)
        distance = np.abs(contacts_rel[:, None] - z_rel[None, :])
        raw_kernel = 1.0 / (distance + float(distance_floor_frac))
        kernel = raw_kernel / np.maximum(raw_kernel.sum(axis=1, keepdims=True), 1e-12)

        row_sum_errors.append(float(np.max(np.abs(kernel.sum(axis=1) - 1.0))))
        kernel_stats[str(area)] = {
            "n_sources": int(idx.size),
            "kernel_min": float(np.min(kernel)),
            "kernel_max": float(np.max(kernel)),
            "kernel_row_sum_max_abs_error": float(np.max(np.abs(kernel.sum(axis=1) - 1.0))),
        }

        # (trial,time,neuron_area) x (contact,neuron_area) -> (trial,time,contact)
        phi = np.einsum("btn,cn->btc", source[:, :, idx], kernel, optimize=True).astype(np.float32)
        lfp_contacts[:, ai] = phi

        if n_contacts > 3:
            interior = (phi[:, :, 2:] - 2.0 * phi[:, :, 1:-1] + phi[:, :, :-2]) / (dz_rel * dz_rel)
            left = (2.0 * phi[:, :, 0:1] - 5.0 * phi[:, :, 1:2] + 4.0 * phi[:, :, 2:3] - phi[:, :, 3:4]) / (dz_rel * dz_rel)
            right = (2.0 * phi[:, :, -1:] - 5.0 * phi[:, :, -2:-1] + 4.0 * phi[:, :, -3:-2] - phi[:, :, -4:-3]) / (dz_rel * dz_rel)
            csd_contacts[:, ai] = -np.concatenate([left, interior, right], axis=2).astype(np.float32)
        elif n_contacts == 3:
            second = (phi[:, :, 2:] - 2.0 * phi[:, :, 1:2] + phi[:, :, 0:1]) / (dz_rel * dz_rel)
            csd_contacts[:, ai] = -np.concatenate([second, second, second], axis=2).astype(np.float32)
        else:
            csd_contacts[:, ai] = 0.0

    if not np.all(np.isfinite(lfp_contacts)):
        raise ValueError("linear-distance lfp_contacts contains non-finite values")
    if not np.all(np.isfinite(csd_contacts)):
        raise ValueError("linear-distance csd_contacts contains non-finite values")

    out = dict(trials)
    out["lfp_contacts"] = lfp_contacts
    out["csd_contacts"] = csd_contacts
    out["contact_depths_m"] = contact_depths_m
    out["lfp_projection"] = {
        "mode": "linear_inverse_distance_depth_proxy",
        "formula": "kernel[c,n] = 1 / (abs(contact_rel[c] - neuron_depth_rel[n]) + floor); row-normalized over n",
        "depth_extension_fraction": float(extension_frac),
        "distance_floor_fraction": float(distance_floor_frac),
        "contact_depth_min_m": float(contact_depths_m.min()),
        "contact_depth_max_m": float(contact_depths_m.max()),
        "contact_depth_min_rel": float(contacts_rel.min()),
        "contact_depth_max_rel": float(contacts_rel.max()),
        "covers_cortex_plus_20pct": bool(
            contact_depths_m.min() <= -0.20 * cz_m + 1e-12
            and contact_depths_m.max() >= 1.20 * cz_m - 1e-12
        ),
        "kernel_row_sum_max_abs_error": float(max(row_sum_errors) if row_sum_errors else 0.0),
        "area_kernel_stats": kernel_stats,
        "field_solver_status": "laminar_proxy_no_pde",
        "physical_amplitude_claim_allowed": False,
    }
    return out


trials = reproject_trials_lfp_linear_distance(
    trials,
    model,
    cfg,
    extension_frac=LFP_DEPTH_EXTENSION_FRAC,
    distance_floor_frac=LFP_DISTANCE_FLOOR_FRAC,
)

print("LFP projection:", trials["lfp_projection"]["mode"])
print(
    "contact depth range (um):",
    round(float(trials["contact_depths_m"].min()) * 1e6, 1),
    "to",
    round(float(trials["contact_depths_m"].max()) * 1e6, 1),
)
print("expected relative coverage:", -LFP_DEPTH_EXTENSION_FRAC, "to", 1.0 + LFP_DEPTH_EXTENSION_FRAC)
print("kernel row-sum max abs error:", trials["lfp_projection"]["kernel_row_sum_max_abs_error"])
assert trials["lfp_projection"]["covers_cortex_plus_20pct"]


In [ ]:
# ── Firing-rate stability report ──────────────────────────────────────────────
# Target regime: average ~5-10 Hz/neuron, peak < ~40 Hz. If a population is too
# hot, lower its *_DRIVE (or *_NOISE); if silent, raise it.
_per_neuron_hz = trials["spikes"].mean(axis=(0, 1)) * 1000.0 / cfg.dt_ms  # (n_neurons,)
_ct = model["neurons"]["cell_type"].to_numpy()
rate_report = {"ALL": {"mean_hz": float(_per_neuron_hz.mean()),
                       "max_hz": float(_per_neuron_hz.max())}}
for _t in cfg.cell_types:
    _idx = np.flatnonzero(_ct == _t)
    if _idx.size:
        rate_report[_t] = {"mean_hz": float(_per_neuron_hz[_idx].mean()),
                           "max_hz": float(_per_neuron_hz[_idx].max())}

print(f"{'pop':>5} | {'mean Hz':>8} | {'max Hz':>7}")
for _k, _v in rate_report.items():
    flag = "  <-- hot (>40 Hz)" if _v["max_hz"] > 40.0 else ""
    print(f"{_k:>5} | {_v['mean_hz']:8.2f} | {_v['max_hz']:7.1f}{flag}")

if rate_report["ALL"]["max_hz"] > 40.0:
    print("\nNote: peak rate exceeds ~40 Hz — lower the hot population's *_DRIVE/*_NOISE.")
elif not (3.0 <= rate_report["ALL"]["mean_hz"] <= 15.0):
    print("\nNote: mean rate is outside the ~5-10 Hz stable band — adjust *_DRIVE.")
else:
    print("\nStable regime: mean in band and peak < 40 Hz.")


## 10. Activity suite (raster / Vm / extracellular proxy / CSD-like / PSD)

In [ ]:
fig_activity = jtfne.vis.activity_trace_suite(
    trials, cfg,
    stage="initial",
    psd_freq_range_hz=(1.0, 150.0),
    psd_log_x=True,
    output_png=f"{cfg.output_dir}/activity_suite.png",
)
fig_activity


## 11. Spectrolaminar relative power and crossing gate

The alpha-beta and gamma profiles should cross across depth. Edit the band ranges if you want to teach a different contrast. The gate below reports the crossing depth for every area and asserts that each area has a band-profile crossing near L4.

In [ ]:
ALPHA_BETA_RANGE_HZ = (10.0, 25.0)
GAMMA_RANGE_HZ = (40.0, 150.0)

# The spectrolaminar cross is EMERGENT from the real Izhikevich dynamics +
# depth-dependent leadfield — it is not imposed. Whether (and where) the
# alpha-beta / gamma profiles cross depends on CELL_TYPE_IZH, LAYER_CELL_TYPE_FRAC,
# density, and drive. Crossing is REPORTED, not hard-required, by default.
REQUIRE_BAND_PROFILE_CROSSING = False
CROSS_TOLERANCE_M = 0.35 * cfg.cz_m

scores, specs = jtfne.tutorial_utils.summarize_spectrolaminar_similarity(
    trials, cfg, areas=cfg.areas,
    alpha_beta_range_hz=ALPHA_BETA_RANGE_HZ,
    gamma_range_hz=GAMMA_RANGE_HZ,
)

# ── Numeric motif checks (objective 7) ───────────────────────────────────────
motif_metrics = {}
for area, spec in specs.items():
    ab = np.asarray(spec["alpha_beta"], dtype=float)
    gm = np.asarray(spec["gamma"],      dtype=float)
    y  = np.asarray(spec["pos_from_l4"], dtype=float)
    rp = np.asarray(spec["relative_power"], dtype=float)

    # 1. Finiteness — all profile arrays must be finite
    assert np.all(np.isfinite(ab)), f"{area}: alpha_beta profile contains non-finite"
    assert np.all(np.isfinite(gm)), f"{area}: gamma profile contains non-finite"
    assert np.all(np.isfinite(rp)), f"{area}: relative_power contains non-finite"

    # 2. Band-peak depths
    ab_peak_depth_um = float(y[int(np.argmax(ab))]) * 1e6   # positive = below L4
    gm_peak_depth_um = float(y[int(np.argmax(gm))]) * 1e6

    # 3. Crossing detection (linear interpolation of the zero crossing nearest L4)
    delta = ab - gm
    sign_changes = np.where(np.signbit(delta[:-1]) != np.signbit(delta[1:]))[0]
    if sign_changes.size > 0:
        i = int(sign_changes[np.argmin(np.abs(y[sign_changes]))])
        y0, y1, d0, d1 = y[i], y[i+1], delta[i], delta[i+1]
        cross_y_um = float(y0 - d0 * (y1 - y0) / (d1 - d0 + 1e-12)) * 1e6
        x_cross_detected = True
    else:
        i = int(np.argmin(np.abs(delta)))
        cross_y_um = float(y[i]) * 1e6
        x_cross_detected = False

    motif_metrics[area] = {
        "ab_peak_depth_um":   round(ab_peak_depth_um, 1),
        "gm_peak_depth_um":   round(gm_peak_depth_um, 1),
        "x_cross_detected":   x_cross_detected,
        "cross_y_um":         round(cross_y_um, 1),
        "motif_direction_ok": bool(gm_peak_depth_um < ab_peak_depth_um),  # gamma superficial
        "lfp_finite":         bool(np.isfinite(trials["lfp_contacts"]).all()),
        "csd_finite":         bool(np.isfinite(trials["csd_contacts"]).all()),
        "similarity_percent": float(spec.get("similarity_percent", float("nan"))),
    }

cross_rows = [{"area": a, "crosses": m["x_cross_detected"], "cross_y_um": m["cross_y_um"]}
              for a, m in motif_metrics.items()]

# Assert finiteness always; soft-assert motif direction (depends on composition)
assert np.isfinite(scores["similarity_percent"].to_numpy()).all()
assert all(m["lfp_finite"] and m["csd_finite"] for m in motif_metrics.values())

if REQUIRE_BAND_PROFILE_CROSSING:
    assert all(m["x_cross_detected"] and abs(m["cross_y_um"]) <= 1e6 * CROSS_TOLERANCE_M
               for m in motif_metrics.values())

display(scores)
try:
    import pandas as pd
    display(pd.DataFrame(motif_metrics).T.style.format({
        "ab_peak_depth_um": "{:.1f}", "gm_peak_depth_um": "{:.1f}",
        "cross_y_um": "{:.1f}", "similarity_percent": "{:.1f}",
    }))
except ImportError:
    for area, m in motif_metrics.items():
        print(area, m)


### Crossing table and spectrolaminar figures

In [ ]:
try:
    import pandas as pd
    display(pd.DataFrame(cross_rows))
except ImportError:
    print(cross_rows)

# Spectrolaminar motif per area: A cell distribution, B relative power spectrum,
# C alpha-beta / gamma crossing. Light theme matches the reference motif.
figs_spectro = jtfne.vis.spectrolaminar_suite_3panel(
    specs, model, cfg, stage="initial", output_dir=cfg.output_dir, theme="light"
)
print("spectrolaminar figures:", list(figs_spectro.keys()))


## 12. Export artifacts (manifest / metrics / validation)

In [ ]:
import json as _json

manifest = cfg.to_manifest_dict()
metrics = {
    "areas": list(cfg.areas),
    "similarity_percent": {
        row["area"]: float(row["similarity_percent"]) for row in scores.to_dict("records")
    },
    "spectrolaminar_crossing": cross_rows,
    "spectrolaminar_motif": motif_metrics,   # objective 7: band-peak depths, crossing, direction
    "firing_rate_hz": rate_report,           # per-population mean/max firing rate
    "n_neurons": int(len(model["neurons"])),
    "lfp_projection": trials.get("lfp_projection", {}),
}
validation = {
    "cell_dist_shape":            list(cfg.cell_dist.shape),
    "cell_dist_normalized":       bool(np.allclose(cfg.cell_dist.sum(axis=1), 1.0)),
    "spectrolaminar_profiles_cross": bool(all(r["crosses"] for r in cross_rows)),
    "motif_direction_ok":         bool(all(m["motif_direction_ok"] for m in motif_metrics.values())),
    "mean_rate_hz":               float(rate_report["ALL"]["mean_hz"]),
    "max_rate_hz":                float(rate_report["ALL"]["max_hz"]),
    "rate_stable":                bool(rate_report["ALL"]["max_hz"] <= 40.0
                                       and 3.0 <= rate_report["ALL"]["mean_hz"] <= 15.0),
    "lfp_finite":                 bool(np.isfinite(trials["lfp_contacts"]).all()),
    "csd_finite":                 bool(np.isfinite(trials["csd_contacts"]).all()),
    "lfp_projection_mode":        trials.get("lfp_projection", {}).get("mode", "unknown"),
    "lfp_covers_cortex_plus_20pct": bool(trials.get("lfp_projection", {}).get("covers_cortex_plus_20pct", False)),
    "lfp_contact_depth_min_m":    float(np.min(trials["contact_depths_m"])),
    "lfp_contact_depth_max_m":    float(np.max(trials["contact_depths_m"])),
    "truth_gates":                cfg.truth_gates,
}

# Strict JSON-safe check — will raise on NaN / Inf / non-serializable values
_json.dumps(metrics, allow_nan=False)
_json.dumps(validation, allow_nan=False)

paths = jtfne.tutorial_utils.export_tutorial_artifacts(
    cfg, manifest_dict=manifest, metrics_dict=metrics,
    validation_dict=validation, output_dir=cfg.output_dir,
)
print("Exported:")
for name, p in paths.items():
    print(f"  {name}: {p}")


## 13. AGSDR fine-tuning (target firing rate + minimize synchrony)

`tune_laminar_agsdr` applies the AGSDR algorithm (adaptive greedy selection / deselection with stochastic restart) to the control knobs, scoring each candidate with the **real** population firing rate and kappa synchrony from short simulations:

`loss = |rate − target| / target + kappa_weight · |kappa|`

Edit the target, the kappa weight, or the tunable `parameters` ranges. This tunes the scaffold's proxy control; it is distinct from `jtfne.agsdr` + `Model.tune` (which optimizes the core-engine Model).

In [ ]:
TUNE_TARGET_RATE_HZ = 5.0
TUNE_KAPPA_WEIGHT = 1.0
tune_gens, tune_pop = (2, 2) if SMOKE else (8, 6)

tuned_control, tune_history = jtfne.tutorial_utils.tune_laminar_agsdr(
    model, cfg,
    target_rate_hz=TUNE_TARGET_RATE_HZ,
    kappa_weight=TUNE_KAPPA_WEIGHT,
    parameters={
        "local_exc_gain": (0.3, 2.5),
        "local_inh_gain": (0.3, 2.5),
        "feedforward_gain": (0.0, 3.0),
    },
    generations=tune_gens, population_size=tune_pop,
    tune_duration_ms=min(cfg.duration_ms, 150.0),
    seed=cfg.seed,
)

try:
    display(tune_history)
except NameError:
    print(tune_history)
print("Tuned control:", {k: round(tuned_control[k], 3)
                          for k in ("local_exc_gain", "local_inh_gain", "feedforward_gain")})
# To use it downstream:  trials = jtfne.tutorial_utils.simulate_laminar_trials(
#     model, cfg, tuned_control, stimulus, target_cells, n_trials=cfg.n_trials)


In [ ]:
#@title Visualize tuned spectrolaminar suite

# 1. Run full trials using the tuned AGSDR control
trials_tuned = jtfne.tutorial_utils.simulate_laminar_trials(
    model,
    cfg,
    tuned_control,
    stimulus,
    target_cells,
    n_trials=cfg.n_trials,
)

trials_tuned = reproject_trials_lfp_linear_distance(
    trials_tuned,
    model,
    cfg,
    extension_frac=LFP_DEPTH_EXTENSION_FRAC,
    distance_floor_frac=LFP_DISTANCE_FLOOR_FRAC,
)

# 2. Recompute spectrolaminar summaries from tuned trials
scores_tuned, specs_tuned = jtfne.tutorial_utils.summarize_spectrolaminar_similarity(
    trials_tuned,
    cfg,
    areas=cfg.areas,
    alpha_beta_range_hz=ALPHA_BETA_RANGE_HZ,
    gamma_range_hz=GAMMA_RANGE_HZ,
)

display(scores_tuned)

# 3. Recompute crossing rows for the tuned condition
cross_rows_tuned = []
motif_metrics_tuned = {}

for area, spec in specs_tuned.items():
    ab = np.asarray(spec["alpha_beta"], dtype=float)
    gm = np.asarray(spec["gamma"], dtype=float)
    y  = np.asarray(spec["pos_from_l4"], dtype=float)
    rp = np.asarray(spec["relative_power"], dtype=float)

    assert np.all(np.isfinite(ab)), f"{area}: tuned alpha_beta profile contains non-finite"
    assert np.all(np.isfinite(gm)), f"{area}: tuned gamma profile contains non-finite"
    assert np.all(np.isfinite(rp)), f"{area}: tuned relative_power contains non-finite"

    ab_peak_depth_um = float(y[int(np.argmax(ab))]) * 1e6
    gm_peak_depth_um = float(y[int(np.argmax(gm))]) * 1e6

    delta = ab - gm
    sign_changes = np.where(np.signbit(delta[:-1]) != np.signbit(delta[1:]))[0]

    if sign_changes.size > 0:
        i = int(sign_changes[np.argmin(np.abs(y[sign_changes]))])
        y0, y1, d0, d1 = y[i], y[i + 1], delta[i], delta[i + 1]
        cross_y_um = float(y0 - d0 * (y1 - y0) / (d1 - d0 + 1e-12)) * 1e6
        x_cross_detected = True
    else:
        i = int(np.argmin(np.abs(delta)))
        cross_y_um = float(y[i]) * 1e6
        x_cross_detected = False

    motif_metrics_tuned[area] = {
        "ab_peak_depth_um": round(ab_peak_depth_um, 1),
        "gm_peak_depth_um": round(gm_peak_depth_um, 1),
        "x_cross_detected": x_cross_detected,
        "cross_y_um": round(cross_y_um, 1),
        "motif_direction_ok": bool(gm_peak_depth_um < ab_peak_depth_um),
        "lfp_finite": bool(np.isfinite(trials_tuned["lfp_contacts"]).all()),
        "csd_finite": bool(np.isfinite(trials_tuned["csd_contacts"]).all()),
        "similarity_percent": float(spec.get("similarity_percent", float("nan"))),
    }

    cross_rows_tuned.append({
        "area": area,
        "crosses": x_cross_detected,
        "cross_y_um": round(cross_y_um, 1),
        "ab_peak_depth_um": round(ab_peak_depth_um, 1),
        "gm_peak_depth_um": round(gm_peak_depth_um, 1),
        "motif_direction_ok": bool(gm_peak_depth_um < ab_peak_depth_um),
    })

try:
    import pandas as pd
    display(pd.DataFrame(cross_rows_tuned))
except ImportError:
    print(cross_rows_tuned)

# 4. Plot the tuned 3-panel spectrolaminar suite
figs_spectro_tuned = jtfne.vis.spectrolaminar_suite_3panel(
    specs_tuned,
    model,
    cfg,
    stage="tuned",
    output_dir=cfg.output_dir,
    theme="dark",   # change to "dark" if supported by your installed version
)

print("tuned spectrolaminar figures:", list(figs_spectro_tuned.keys()))

## 14. Knock-out experiment

Lesion a layer / cell type / area and measure the downstream effect. Edit `KNOCKOUT` to silence any population (e.g. the feedforward source `V1 L2/3 E`, or thalamorecipient `V4 L4`). The network is re-simulated with those neurons silenced and per-area firing rates are compared to the intact run.

In [ ]:
import dataclasses as _dc

KNOCKOUT = {"area": "V1", "layers": ("L2", "L3"), "cell_types": ("E",)}  # edit target

cfg_lesion = _dc.replace(cfg, lesion_spec=(KNOCKOUT,))
trials_lesion = jtfne.tutorial_utils.simulate_laminar_trials(
    model, cfg_lesion, cfg.base_control, stimulus, target_cells, n_trials=cfg.n_trials
)

def _area_rate(tr, area):
    idx = np.flatnonzero(model["neurons"]["area"].to_numpy() == area)
    return float(tr["spikes"][:, :, idx].mean()) * 1000.0 / cfg.dt_ms

print(f"Knocked out {trials_lesion['n_lesioned']} neurons: {KNOCKOUT}\n")
print(f"{'area':>6} | {'intact (Hz)':>12} | {'lesioned (Hz)':>13} | {'delta':>7}")
for area in cfg.areas:
    ri, rl = _area_rate(trials, area), _area_rate(trials_lesion, area)
    print(f"{area:>6} | {ri:12.2f} | {rl:13.2f} | {ri - rl:+7.2f}")


---
**Done.** Edit the controls above to change model scale, duration, column dimensions, layer/type ratios, stimulus, and spectrolaminar bands. Heavy computation stays in `jaxfne.tutorial_utils` and `jaxfne.vis`.
